# Zero-Day Attack Classification — WUSTL-IIoT  (Anonymized Ablation)

Same zero-day experiment as `04-zero-day-detection-claude.ipynb`, but run **twice**:

1. **Original**: real feature names (`SrcPkts`, `DstRate`, …)
2. **Anonymized**: opaque labels `f0`, `f1`, … `f{n}`

**Hypothesis:** If feature-name priors drive zero-day generalisation, anonymizing names should collapse ZDR to DT/RF levels (~0–10%).

In [1]:
################################################################################
# Cell 0 - Configuration
################################################################################

# === Change WITHHELD_CLASS to run a different zero-day scenario ===
#
# Candidate classes for WUSTL-IIoT (sample counts):
#   'Reconn'   8,240 samples - RECOMMENDED (distinct from dominant DoS profile, adequate size)
#   'DoS'     78,305 samples - high-support stress test for generalisation
#   'CommInj'    259 samples - low support; use only for exploratory runs
#   'Backdoor'   212 samples - low support; use only for exploratory runs
#
# Selection rationale:
# Reconn provides a meaningful zero-day class with enough samples while differing
# from the dominant DoS attack behavior in the known-attack training pool.

WITHHELD_CLASS  = 'Reconn'

DATASET_NAME    = 'wustl-iiot'
LABEL_COL       = 'Traffic'      # multiclass label column in population CSV
BENIGN_CLASS    = 'normal'       # benign class name

# Columns to exclude from features (label columns)
DROP_COLS       = ['Target', 'Traffic']

N_REPR          = 10       # representative samples per class fed to LLM
MAX_EMBED       = 100      # max rows to embed per class
k               = 5        # number of rules
n               = 5        # number of feedback iterations
SEEDS           = [42, 123, 456]
SEED_MAIN       = 42

print(f'Withheld (zero-day) class : {WITHHELD_CLASS}')
print(f'Benign class              : {BENIGN_CLASS}')
print(f'Label column              : {LABEL_COL}')
print(f'Rules: {k} | Iterations: {n} | Seeds: {SEEDS}')

Withheld (zero-day) class : Reconn
Benign class              : normal
Label column              : Traffic
Rules: 5 | Iterations: 5 | Seeds: [42, 123, 456]


In [ ]:
################################################################################
# Cell 0b — Anonymization map
#
# Built AFTER feature_cols is known (Cell 1 loads data and sets feature_cols).
# anon_map   : real_name  → f{i}
# reverse_map: f{i}       → real_name
################################################################################

# feature_cols is defined in Cell 2 (load) — run that first, then come back
# if running out of order. Here we define a deferred builder:
def build_anon_maps(cols):
    a = {name: f'f{i}' for i, name in enumerate(cols)}
    r = {f'f{i}': name for i, name in enumerate(cols)}
    return a, r

print('build_anon_maps() ready — call after Cell 2 has set feature_cols.')

build_anon_maps() ready — call after Cell 2 has set feature_cols.


In [3]:
################################################################################
# Cell 1 - Load data
#
# Single source: wustl-iiot-population.csv (full dataset with original class labels)
# All splits (training, test, zero-day) are derived from this file only.
################################################################################

import pandas as pd
import numpy as np
import os
from tabulate import tabulate


DATA_PATH = '/Users/S4160163/Documents/Projects/RAG Paper/data/wustl-iiot/wustl-iiot-population.csv'
df_raw = pd.read_csv(DATA_PATH)

# Keep only numeric feature columns (drop label columns)
feature_cols = [
    c for c in df_raw.select_dtypes(include=[np.number]).columns
    if c not in DROP_COLS
]
df = df_raw[feature_cols + [LABEL_COL]].copy()

print(f'=== WUSTL-IIoT Population ===')
print(f'Data path  : {DATA_PATH}')
print(f'Total rows : {len(df):,}')
print(f'Features   : {len(feature_cols)}  ->  {feature_cols}')
print()

label_counts = df[LABEL_COL].value_counts()
print('Class distribution:')
print(label_counts.to_string())
print()

benign_count   = label_counts.get(BENIGN_CLASS, 0)
withheld_count = label_counts.get(WITHHELD_CLASS, 0)
print(f'Benign ("{BENIGN_CLASS}") rows   : {benign_count:,}')
print(f'Withheld ("{WITHHELD_CLASS}") rows: {withheld_count:,}')
print(f'\nTraining will be balanced to the smaller class size (1:1).')

=== WUSTL-IIoT Population ===
Data path  : /Users/S4160163/Documents/Projects/RAG Paper/data/wustl-iiot/wustl-iiot-population.csv
Total rows : 1,194,464
Features   : 43  ->  ['Mean', 'Sport', 'Dport', 'SrcPkts', 'DstPkts', 'TotPkts', 'DstBytes', 'SrcBytes', 'TotBytes', 'SrcLoad', 'DstLoad', 'Load', 'SrcRate', 'DstRate', 'Rate', 'SrcLoss', 'DstLoss', 'Loss', 'pLoss', 'SrcJitter', 'DstJitter', 'SIntPkt', 'DIntPkt', 'Proto', 'Dur', 'TcpRtt', 'IdleTime', 'Sum', 'Min', 'Max', 'sDSb', 'sTtl', 'dTtl', 'sIpId', 'dIpId', 'SAppBytes', 'DAppBytes', 'TotAppByte', 'SynAck', 'RunTime', 'sTos', 'SrcJitAct', 'DstJitAct']

Class distribution:
Traffic
normal      1107448
DoS           78305
Reconn         8240
CommInj         259
Backdoor        212

Benign ("normal") rows   : 1,107,448
Withheld ("Reconn") rows: 8,240

Training will be balanced to the smaller class size (1:1).


In [4]:
################################################################################
# Cell 2 — Prepare zero-day split
#
# Balancing strategy:
#   - Downsample the larger class so benign and known attacks are exactly matched
#   - Use a 1:1 pool to avoid imbalance-driven bias and reduce runtime
#   - Withheld class is excluded from the training pool entirely
#
# Zero-day test set:
#   - ALL rows of WITHHELD_CLASS (never in training, never in test_known)
################################################################################

benign_df    = df[df[LABEL_COL] == BENIGN_CLASS].copy()
known_atk_df = df[
    (df[LABEL_COL] != BENIGN_CLASS) & (df[LABEL_COL] != WITHHELD_CLASS)
].copy()
zeroday_df   = df[df[LABEL_COL] == WITHHELD_CLASS].copy()

print(f'Benign rows   : {len(benign_df):,}')
print(f'Known attack  : {len(known_atk_df):,}  ({known_atk_df[LABEL_COL].nunique()} classes: {sorted(known_atk_df[LABEL_COL].unique())})')
print(f'Zero-day rows : {len(zeroday_df):,}  ("{WITHHELD_CLASS}")')

# Balance: sample both classes to the same target count (1:1)
target_count = min(len(benign_df), len(known_atk_df))
benign_sampled = benign_df.sample(n=target_count, random_state=SEED_MAIN)
known_atk_sampled = known_atk_df.sample(n=target_count, random_state=SEED_MAIN)
print(f'\nBalanced pool: {target_count:,} benign + {target_count:,} known attack (1:1)')

# Binary labels
train_pool = pd.concat([benign_sampled, known_atk_sampled], ignore_index=True)
train_pool['binary_label'] = train_pool[LABEL_COL].apply(
    lambda x: 'normal' if x == BENIGN_CLASS else 'attack'
)

# Stratified 80/20 split
from sklearn.model_selection import train_test_split
train_idx, test_idx = train_test_split(
    train_pool.index, test_size=0.2, random_state=SEED_MAIN,
    stratify=train_pool['binary_label']
)
train_df      = train_pool.loc[train_idx]
test_known_df = train_pool.loc[test_idx]

# Feature-only dataframes (global — referenced by evaluation_tool and evaluate_node)
normal_df_train = train_df[train_df['binary_label'] == 'normal'][feature_cols].reset_index(drop=True)
attack_df_train = train_df[train_df['binary_label'] == 'attack'][feature_cols].reset_index(drop=True)
normal_df_test  = test_known_df[test_known_df['binary_label'] == 'normal'][feature_cols].reset_index(drop=True)
attack_df_test  = test_known_df[test_known_df['binary_label'] == 'attack'][feature_cols].reset_index(drop=True)

# Zero-day test = ALL withheld class rows (guaranteed absent from training)
zeroday_df_test = zeroday_df[feature_cols].reset_index(drop=True)

data = [
    ['Normal (train)',                    len(normal_df_train)],
    ['Known attack (train)',              len(attack_df_train)],
    ['Normal (test, known)',              len(normal_df_test)],
    ['Known attack (test, known)',        len(attack_df_test)],
    [f'Zero-day test ({WITHHELD_CLASS})', len(zeroday_df_test)],
]
print('\nDataset splits:')
print(tabulate(data, headers=['Split', 'Count'], tablefmt='grid'))
print(f'\nFeatures : {len(feature_cols)}')
print('Zero-day samples are NOT present in training.')

Benign rows   : 1,107,448
Known attack  : 78,776  (3 classes: ['Backdoor', 'CommInj', 'DoS'])
Zero-day rows : 8,240  ("Reconn")

Balanced pool: 78,776 benign + 78,776 known attack (1:1)

Dataset splits:
+----------------------------+---------+
| Split                      |   Count |
+============================+=========+
| Normal (train)             |   63021 |
+----------------------------+---------+
| Known attack (train)       |   63020 |
+----------------------------+---------+
| Normal (test, known)       |   15755 |
+----------------------------+---------+
| Known attack (test, known) |   15756 |
+----------------------------+---------+
| Zero-day test (Reconn)     |    8240 |
+----------------------------+---------+

Features : 43
Zero-day samples are NOT present in training.


In [5]:
################################################################################
# Cell 3 — Representative samples via BGE-M3 embeddings
#
# Identical methodology to other zero-day notebooks:
#   1. Subsample up to MAX_EMBED rows per class
#   2. Embed as stringified row lists using BAAI/bge-m3
#   3. Compute mean embedding
#   4. Select top N_REPR rows by cosine similarity to mean
################################################################################

import json
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from tqdm import tqdm

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'mps'},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 64}
)


def get_representative_samples_bge(
    df: pd.DataFrame, n: int = 10, max_embed: int = 100, seed: int = 42
) -> pd.DataFrame:
    """
    Subsample up to max_embed rows, embed using BGE-M3, compute mean embedding,
    return n rows with highest cosine similarity to the mean.
    """
    sample = df.sample(n=min(max_embed, len(df)), random_state=seed)
    docs   = [str(row.tolist()) for _, row in sample.iterrows()]
    vecs   = np.array(embeddings.embed_documents(docs))
    mean_vec = vecs.mean(axis=0)
    norms = np.linalg.norm(vecs, axis=1) * np.linalg.norm(mean_vec)
    sims  = (vecs @ mean_vec) / np.where(norms == 0, 1e-9, norms)
    top_idx = np.argsort(sims)[::-1][:n]
    return sample.iloc[top_idx]


print('Computing normal representative samples via BGE-M3...')
normal_repr = get_representative_samples_bge(
    normal_df_train, n=N_REPR, max_embed=MAX_EMBED, seed=SEED_MAIN
)

print('Computing attack representative samples via BGE-M3 (known attacks only)...')
attack_repr = get_representative_samples_bge(
    attack_df_train, n=N_REPR, max_embed=MAX_EMBED, seed=SEED_MAIN
)

normal_entries_dict = {col: normal_repr[col].tolist() for col in feature_cols}
attack_entries_dict = {col: attack_repr[col].tolist() for col in feature_cols}
normal_entries = json.dumps(normal_entries_dict)
attack_entries = json.dumps(attack_entries_dict)

print(f'\nRepresentative normal  samples : {len(normal_repr)}')
print(f'Representative attack  samples : {len(attack_repr)}')
print(f'Embedding model        : BAAI/bge-m3 (normalised cosine, batch=64)')
print(f'Max embedded per class : {MAX_EMBED}')

Computing normal representative samples via BGE-M3...
Computing attack representative samples via BGE-M3 (known attacks only)...

Representative normal  samples : 10
Representative attack  samples : 10
Embedding model        : BAAI/bge-m3 (normalised cosine, batch=64)
Max embedded per class : 100


In [6]:
################################################################################
# Cell 3b — Finalize anonymization maps and build entry dicts
#
# Produces four entry strings:
#   normal_entries_orig / attack_entries_orig  — real feature names
#   normal_entries_anon / attack_entries_anon  — anonymized f0…fn labels
################################################################################

anon_map, reverse_map = build_anon_maps(feature_cols)

normal_entries_orig = json.dumps({col: normal_repr[col].tolist() for col in feature_cols})
attack_entries_orig = json.dumps({col: attack_repr[col].tolist() for col in feature_cols})

normal_entries_anon = json.dumps({anon_map[col]: normal_repr[col].tolist() for col in feature_cols})
attack_entries_anon = json.dumps({anon_map[col]: attack_repr[col].tolist() for col in feature_cols})

print(f'Anonymization map built: {len(anon_map)} features')
print(f'Sample mapping: {list(anon_map.items())[:5]}')

Anonymization map built: 43 features
Sample mapping: [('Mean', 'f0'), ('Sport', 'f1'), ('Dport', 'f2'), ('SrcPkts', 'f3'), ('DstPkts', 'f4')]


In [7]:
################################################################################
# Cell 4 — Anon-aware Policy Evaluation Tool
#
# Accepts both real names and f{i} labels via reverse_map lookup.
################################################################################

import operator as op_module
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from statistics import mode
from typing import Annotated
from langchain_core.tools import tool

show_progress = False
operators = {
    '<':  op_module.lt,
    '>':  op_module.gt,
    '==': op_module.eq,
    '<=': op_module.le,
    '>=': op_module.ge,
    '!=': op_module.ne,
}


@tool
def evaluation_tool(
        feature_name: Annotated[str, 'Feature name (real or anonymous f{i})'],
        value: Annotated[float, 'Threshold value'],
        op: Annotated[str, 'Operator (<, >, <=, >=, ==, !=']
) -> float:
    """Evaluate a single threshold rule on training data. Returns macro F1-score."""
    real_name = reverse_map.get(feature_name, feature_name)
    try:
        value = float(value)
    except (ValueError, TypeError):
        pass
    if op not in operators:
        raise ValueError(f'Unsupported operator: {op}')
    datasets = {'normal': normal_df_train, 'attack': attack_df_train}
    y_pred, y_true = [], []
    for label, dataset in datasets.items():
        for i in tqdm(range(len(dataset)), disable=not show_progress,
                      ncols=100, desc=f'Evaluating {label}...'):
            y_true.append(label)
            try:
                y_pred.append('attack' if operators[op](dataset.iloc[i][real_name], value) else 'normal')
            except (KeyError, TypeError):
                y_pred.append('normal')
    report = classification_report(y_true, y_pred, digits=4, output_dict=True)
    return report['macro avg']['f1-score']


print('evaluation_tool (anon-aware) defined.')

evaluation_tool (anon-aware) defined.


In [ ]:
################################################################################
# Cell 5 — LangGraph Pipeline
#
# Note: langgraph.prebuilt (ToolNode) is not available in this version of langgraph.
# A minimal custom ToolNode is implemented below that is functionally equivalent.
################################################################################

import dotenv, os, json
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

dotenv.load_dotenv(os.getcwd() + '/../.env')


class State(MessagesState):
    i: int
    max_f1s: float
    best_tool_calls: list


llm = ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0.1)
# llm = ChatGoogleGenerativeAI(model='gemini-1.5-pro', temperature=0.1)
# llm = ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0.1)

tools_list = [evaluation_tool]
llm_with_tools = llm.bind_tools(tools_list)


def extract_tool_calls(message):
    """Return normalized tool calls from either parsed or raw OpenAI format."""
    parsed = getattr(message, 'tool_calls', None)
    if parsed:
        return parsed
    raw_calls = (getattr(message, 'additional_kwargs', {}) or {}).get('tool_calls', [])
    normalized = []
    for call in raw_calls:
        fn = call.get('function', {})
        args = fn.get('arguments', {})
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except Exception:
                args = {}
        normalized.append({'id': call.get('id', ''), 'name': fn.get('name', ''), 'args': args})
    return normalized


# Custom ToolNode — replaces langgraph.prebuilt.ToolNode (unavailable in this version)
def make_tool_node(tools):
    tools_by_name = {t.name: t for t in tools}
    def tool_node(state):
        ai_msg = next(
            m for m in reversed(state['messages'])
            if extract_tool_calls(m)
        )
        results = []
        for tc in extract_tool_calls(ai_msg):
            result = tools_by_name[tc['name']].invoke(tc['args'])
            results.append(ToolMessage(content=str(result), tool_call_id=tc['id']))
        return {'messages': results}
    return tool_node


def llm_node(state):
    completion = llm_with_tools.invoke(state['messages'])
    return {'messages': [completion], 'i': state['i'] + 1}


def evaluate_node(state):
    """Evaluate current rules on known-attack test set and send feedback to LLM."""
    ai_messages = [
        m for m in state['messages']
        if isinstance(m, AIMessage) and extract_tool_calls(m)
    ]
    if not ai_messages:
        return {}
    tool_calls = extract_tool_calls(ai_messages[-1])

    datasets = {'normal': normal_df_test, 'attack': attack_df_test}
    y_pred, y_true = [], []
    for label, dataset in datasets.items():
        for i in tqdm(range(len(dataset)), disable=False, ncols=100,
                      desc=f'Test eval {label}...'):
            try:
                votes = [
                    'attack' if operators[tc['args']['op']](
                        dataset.iloc[i][
                            reverse_map.get(tc['args']['feature_name'],
                                            tc['args']['feature_name'])],
                        float(tc['args']['value'])
                    ) else 'normal'
                    for tc in tool_calls
                ]
                y_pred.append(mode(votes))
            except Exception:
                y_pred.append('normal')
            y_true.append(label)

    report = classification_report(y_true, y_pred, digits=4, output_dict=True)
    matrix = confusion_matrix(y_true, y_pred)
    f1     = report['macro avg']['f1-score']
    print(f'  Iter {state["i"]}: known-attack test macro-F1 = {f1:.4f}')
    print(matrix)

    new_max  = max(state['max_f1s'], f1)
    new_best = tool_calls if f1 >= state['max_f1s'] else state['best_tool_calls']

    feedback = HumanMessage(
        f'Current macro avg F1 on test set (known attacks only): {f1:.4f}. '
        f'Best so far: {new_max:.4f}. '
        f'If this is greater than the previous best, keep performing rules and revise underperforming ones. '
        f'Otherwise revise all rules to exceed the best. '
        f'Generate exactly {k} rules and make a tool call for each.'
    )
    return {
        'messages': [feedback],
        'max_f1s':  new_max,
        'best_tool_calls': new_best,
    }


def tools_condition_edge(state):
    return 'tools' if extract_tool_calls(state['messages'][-1]) else 'evaluate_node'


def feedback_condition_edge(state):
    return 'llm_node' if state['i'] < n else END


builder = StateGraph(State)
builder.add_node('llm_node', llm_node)
builder.add_node('tools', make_tool_node(tools_list))
builder.add_node('evaluate_node', evaluate_node)

builder.add_edge(START, 'llm_node')
builder.add_conditional_edges('llm_node', tools_condition_edge, ['tools', 'evaluate_node'])
builder.add_edge('tools', 'llm_node')
builder.add_conditional_edges('evaluate_node', feedback_condition_edge, ['llm_node', END])

graph = builder.compile(checkpointer=MemorySaver())
display(Image(graph.get_graph().draw_mermaid_png()))
print('Graph compiled.')

In [9]:
################################################################################
# Cell 6a — Run pipeline: ORIGINAL feature names
################################################################################

import dotenv, os
from langchain_core.messages import SystemMessage, HumanMessage

dotenv.load_dotenv(os.getcwd() + '/../.env')

system_message = SystemMessage(
    f"""You are a skilled security data analyst.
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Your task is to generate exactly {k} simple and deterministic rules for the top {k} important features to filter attack entries.
Supported operators: >, <, >=, <=
NEVER use '==' or '!=' - these are forbidden for numeric features.
Generate exactly {k} rules and make a tool call for each rule."""
)

print('--- ORIGINAL feature names ---')
initial_state_orig = State(
    i=0, max_f1s=0.5, best_tool_calls=[],
    messages=[
        system_message,
        HumanMessage(
            f'Analyze the following network data and generate {k} rules to identify attack entries.\n\n'
            f'Normal Entries:\n```{normal_entries_orig}```\n\n'
            f'Attack Entries:\n```{attack_entries_orig}```'
        )
    ]
)
config_orig = {
    'configurable': {'thread_id': f'anon-orig-{DATASET_NAME}-{WITHHELD_CLASS}-seed{SEED_MAIN}'},
    'recursion_limit': 100
}
graph_orig = builder.compile(checkpointer=MemorySaver())
output_orig = graph_orig.invoke(initial_state_orig, config_orig)

best_tool_calls_orig = output_orig['best_tool_calls']
best_f1_orig = output_orig['max_f1s']
print(f'\nBest known-attack macro-F1 (original): {best_f1_orig:.4f}')
print(f'Rules ({len(best_tool_calls_orig)}):')
for tc in best_tool_calls_orig:
    a = tc['args']
    print(f'  {a["feature_name"]:25s} {a["op"]:2s} {a["value"]}')

--- ORIGINAL feature names ---


Test eval attack...: 100%|█████████████████████████████████| 15756/15756 [00:01<00:00, 12081.69it/s]


  Iter 2: known-attack test macro-F1 = 0.8030
[[12035  3721]
 [ 2477 13278]]


Test eval attack...: 100%|█████████████████████████████████| 15756/15756 [00:01<00:00, 11229.20it/s]


  Iter 4: known-attack test macro-F1 = 0.8030
[[12034  3722]
 [ 2477 13278]]


Test eval attack...: 100%|█████████████████████████████████| 15756/15756 [00:01<00:00, 11981.82it/s]


  Iter 6: known-attack test macro-F1 = 0.8559
[[11548  4208]
 [  260 15495]]

Best known-attack macro-F1 (original): 0.8559
Rules (5):
  SrcPkts                   <  5
  DstPkts                   <= 1
  TotBytes                  <  500
  SrcLoad                   >  300000
  Rate                      >  20000


In [10]:
################################################################################
# Cell 6b — Run pipeline: ANONYMIZED feature names
################################################################################

print('--- ANONYMIZED feature names ---')
initial_state_anon = State(
    i=0, max_f1s=0.5, best_tool_calls=[],
    messages=[
        system_message,
        HumanMessage(
            f'Analyze the following network data and generate {k} rules to identify attack entries.\n\n'
            f'Normal Entries:\n```{normal_entries_anon}```\n\n'
            f'Attack Entries:\n```{attack_entries_anon}```'
        )
    ]
)
config_anon = {
    'configurable': {'thread_id': f'anon-anon-{DATASET_NAME}-{WITHHELD_CLASS}-seed{SEED_MAIN}'},
    'recursion_limit': 100
}
graph_anon = builder.compile(checkpointer=MemorySaver())
output_anon = graph_anon.invoke(initial_state_anon, config_anon)

best_tool_calls_anon = output_anon['best_tool_calls']
best_f1_anon = output_anon['max_f1s']
print(f'\nBest known-attack macro-F1 (anon): {best_f1_anon:.4f}')
print(f'Rules ({len(best_tool_calls_anon)}) — shown with real names:')
for tc in best_tool_calls_anon:
    a = tc['args']
    real = reverse_map.get(a['feature_name'], a['feature_name'])
    print(f'  {a["feature_name"]:6s} ({real:20s}) {a["op"]:2s} {a["value"]}')

--- ANONYMIZED feature names ---


Test eval attack...: 100%|█████████████████████████████████| 15756/15756 [00:01<00:00, 12156.07it/s]


  Iter 2: known-attack test macro-F1 = 0.8548
[[11938  3818]
 [  712 15043]]


Test eval attack...: 100%|█████████████████████████████████| 15756/15756 [00:01<00:00, 12067.88it/s]


  Iter 4: known-attack test macro-F1 = 0.8631
[[11929  3827]
 [  438 15317]]


Test eval attack...: 100%|█████████████████████████████████| 15756/15756 [00:01<00:00, 11009.41it/s]


  Iter 6: known-attack test macro-F1 = 0.8617
[[11781  3975]
 [  324 15431]]

Best known-attack macro-F1 (anon): 0.8631
Rules (5) — shown with real names:
  f1     (Sport               ) <  50000
  f2     (Dport               ) <  502
  f3     (SrcPkts             ) <  10
  f4     (DstPkts             ) <  8
  f9     (SrcLoad             ) >  100000


In [11]:
################################################################################
# Cell 7 — ZDR and FPR_benign for both conditions
################################################################################

from statistics import mode

def evaluate_zero_day_anon(tool_calls, zd_df):
    """Like evaluate_zero_day but resolves anon feature names via reverse_map."""
    preds = []
    for i in tqdm(range(len(zd_df)), ncols=100,
                  desc=f'Zero-day eval [{WITHHELD_CLASS}]...'):
        try:
            votes = [
                'attack' if operators[tc['args']['op']](
                    zd_df.iloc[i][reverse_map.get(tc['args']['feature_name'],
                                                   tc['args']['feature_name'])],
                    float(tc['args']['value'])
                ) else 'normal'
                for tc in tool_calls
            ]
            preds.append(mode(votes))
        except Exception:
            preds.append('normal')
    zdr = sum(p == 'attack' for p in preds) / max(len(preds), 1)
    return zdr, preds


def fpr_and_precision_anon(tool_calls, benign_df, tp_count):
    """Compute FPR_benign and precision_reconn. Resolves anon names."""
    bp = []
    for i in tqdm(range(len(benign_df)), ncols=100, desc='FPR_benign...'):
        try:
            votes = [
                'attack' if operators[tc['args']['op']](
                    benign_df.iloc[i][reverse_map.get(tc['args']['feature_name'],
                                                       tc['args']['feature_name'])],
                    float(tc['args']['value'])
                ) else 'normal'
                for tc in tool_calls
            ]
            bp.append(mode(votes))
        except Exception:
            bp.append('normal')
    fp_b  = sum(p == 'attack' for p in bp)
    fpr_b = fp_b / max(len(benign_df), 1)
    prec  = tp_count / (tp_count + fp_b) if (tp_count + fp_b) > 0 else 0.0
    return fpr_b, prec, fp_b


# Original
zdr_orig, preds_orig = evaluate_zero_day_anon(best_tool_calls_orig, zeroday_df_test)
tp_orig = sum(p == 'attack' for p in preds_orig)
fpr_orig, prec_orig, fp_orig = fpr_and_precision_anon(best_tool_calls_orig, normal_df_test, tp_orig)

# Anonymized
zdr_anon, preds_anon = evaluate_zero_day_anon(best_tool_calls_anon, zeroday_df_test)
tp_anon = sum(p == 'attack' for p in preds_anon)
fpr_anon, prec_anon, fp_anon = fpr_and_precision_anon(best_tool_calls_anon, normal_df_test, tp_anon)

print(f'\n=== Condition Comparison (seed {SEED_MAIN}) ===')
print(f'Condition    ZDR       FPR_benign  Precision_reconn')
print(f'Original     {zdr_orig:.4f}    {fpr_orig:.4f}      {prec_orig:.4f}')
print(f'Anonymized   {zdr_anon:.4f}    {fpr_anon:.4f}      {prec_anon:.4f}')
print(f'\nΔZDR (orig − anon): {zdr_orig - zdr_anon:+.4f}')

FPR_benign...: 100%|███████████████████████████████████████| 15755/15755 [00:01<00:00, 12091.24it/s]


=== Condition Comparison (seed 42) ===
Condition    ZDR       FPR_benign  Precision_reconn
Original     0.8140    0.0165      0.9627
Anonymized   0.9717    0.0278      0.9481

ΔZDR (orig − anon): -0.1578


In [12]:
################################################################################
# Cell 8 — ML baseline comparison
################################################################################

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_train   = train_df[feature_cols].values
y_train   = train_df['binary_label'].values
X_test    = test_known_df[feature_cols].values
y_test    = test_known_df['binary_label'].values
X_zd      = zeroday_df_test.values
X_benign  = normal_df_test.values

ml_results = []
for name, model in [
    ('Decision Tree', DecisionTreeClassifier(random_state=SEED_MAIN)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=SEED_MAIN, n_jobs=-1)),
]:
    model.fit(X_train, y_train)
    rep = classification_report(model.predict(X_test), y_test, output_dict=True)
    known_f1  = rep['macro avg']['f1-score']
    y_zd      = model.predict(X_zd)
    y_ben     = model.predict(X_benign)
    ml_zdr    = (y_zd  == 'attack').mean()
    fp_b      = (y_ben == 'attack').sum()
    fpr_b     = fp_b / max(len(X_benign), 1)
    tp_zd     = (y_zd  == 'attack').sum()
    prec_zd   = tp_zd / (tp_zd + fp_b) if (tp_zd + fp_b) > 0 else 0.0
    ml_results.append([name, f'{known_f1:.4f}', f'{ml_zdr:.4f}', f'{fpr_b:.4f}', f'{prec_zd:.4f}'])
    print(f'{name:20s}  ZDR={ml_zdr:.4f}  FPR_benign={fpr_b:.4f}  Prec_reconn={prec_zd:.4f}')

Decision Tree         ZDR=0.0001  FPR_benign=0.0001  Prec_reconn=0.5000
Random Forest         ZDR=0.0579  FPR_benign=0.0002  Prec_reconn=0.9938


In [13]:
################################################################################
# Cell 9 — Full comparison table + save
################################################################################

import datetime, os
from tabulate import tabulate

all_results = [
    ['LLM (original names)',  f'{best_f1_orig:.4f}', f'{zdr_orig:.4f}  ({zdr_orig*100:.1f}%)',
     f'{fpr_orig:.4f}  ({fpr_orig*100:.1f}%)', f'{prec_orig:.4f}'],
    ['LLM (anonymized)',      f'{best_f1_anon:.4f}', f'{zdr_anon:.4f}  ({zdr_anon*100:.1f}%)',
     f'{fpr_anon:.4f}  ({fpr_anon*100:.1f}%)', f'{prec_anon:.4f}'],
] + ml_results

print(f'\n=== ANONYMIZED ZERO-DAY EXPERIMENT RESULTS ===')
print(f'Dataset  : {DATASET_NAME}')
print(f'Withheld : {WITHHELD_CLASS}  ({len(zeroday_df_test):,} zero-day test samples)')
print(f'Seed     : {SEED_MAIN}')
print()
print(tabulate(
    all_results,
    headers=['Method', 'Known-Attack F1', f'ZDR — {WITHHELD_CLASS}', 'FPR_benign', 'Precision_reconn'],
    tablefmt='grid'
))

os.makedirs('results/anon', exist_ok=True)
ts = datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S')
out = {
    'dataset': DATASET_NAME, 'withheld_class': WITHHELD_CLASS, 'seed': SEED_MAIN,
    'zero_day_samples': len(zeroday_df_test),
    'original':   {'known_f1': best_f1_orig, 'zdr': zdr_orig, 'fpr_benign': fpr_orig,
                   'precision_reconn': prec_orig,
                   'rules': [tc['args'] for tc in best_tool_calls_orig]},
    'anonymized': {'known_f1': best_f1_anon, 'zdr': zdr_anon, 'fpr_benign': fpr_anon,
                   'precision_reconn': prec_anon,
                   'rules': [{'feature_name': tc['args']['feature_name'],
                               'real_name': reverse_map.get(tc['args']['feature_name'], tc['args']['feature_name']),
                               'value': tc['args']['value'], 'op': tc['args']['op']}
                              for tc in best_tool_calls_anon]},
    'delta_zdr': zdr_orig - zdr_anon,
}
out_path = f'results/anon/zeroday-{WITHHELD_CLASS}-anon-ablation-seed{SEED_MAIN}-{ts}.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'\nResults saved to {out_path}')
print(f'\nΔZDR (original − anonymized): {zdr_orig - zdr_anon:+.4f}')


=== ANONYMIZED ZERO-DAY EXPERIMENT RESULTS ===
Dataset  : wustl-iiot
Withheld : Reconn  (8,240 zero-day test samples)
Seed     : 42

+----------------------+-------------------+-----------------+----------------+--------------------+
| Method               |   Known-Attack F1 | ZDR — Reconn    | FPR_benign     |   Precision_reconn |
+======================+===================+=================+================+====================+
| LLM (original names) |            0.8559 | 0.8140  (81.4%) | 0.0165  (1.7%) |             0.9627 |
+----------------------+-------------------+-----------------+----------------+--------------------+
| LLM (anonymized)     |            0.8631 | 0.9717  (97.2%) | 0.0278  (2.8%) |             0.9481 |
+----------------------+-------------------+-----------------+----------------+--------------------+
| Decision Tree        |            0.9999 | 0.0001          | 0.0001         |             0.5    |
+----------------------+-------------------+--------------